<a href="https://colab.research.google.com/github/zerogradlabs/zerograd-python/blob/main/examples/no-target.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Without a target: anomalies, gaps, synthetic rows and segments

Four tasks with no labels, with tabular foundation models and the open source SDK, locally. Each one runs the classic scikit-learn method next
to the models whenever there is something to measure, so you see what the model adds.

| task | foundation models | against |
|---|---|---|
| `zg.anomalies` | TabPFN v2, TabICLv2, FoMo-0D | Isolation Forest, LOF, kNN |
| `zg.impute` | TabPFN v2, TabICLv2 | mean/mode, KNN, iterative imputation |
| `zg.synthesize` | TabPFN v2, TabICLv2 | sampling each column on its own |
| `zg.cluster` | ZEUS | k-means, Gaussian mixture |

Every model here allows commercial use. TabPFN v2 asks you to show "Built with PriorLabs-TabPFN".

In [1]:
!pip install -q zerograd

In [2]:
import numpy as np
import pandas as pd

import zerograd
from zerograd import ZeroGrad

zg = ZeroGrad()
zerograd.list_tasks()     # every task and which models run it

/home/runner/work/zerograd-python/zerograd-python/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
◆ ZeroGrad 0.2.4 · local · 2 models ready: TabPFN v2, TabICLv2


  Backprop called in sick. We don't need it.


Task,Status,What it does,Commercial models,Non-commercial models
Classificationclassification,In the SDK,"Predict a category: binary (churn, fraud, default) or multiclass (up to 10 classes).","tabpfn-v2, tabicl-v2, mitra-v2, mitra, tabdpt","tabpfn-2.5, tabpfn-2.6, tabpfn-3, tabpfn-3.5, tabfm, exaone-tabular, limix-2"
Regressionregression,In the SDK,"Predict a number (amount, price, demand, time). Runs locally in this version.","tabpfn-v2, tabicl-v2, mitra-v2, mitra, tabdpt","tabpfn-2.5, tabpfn-2.6, tabpfn-3, tabpfn-3.5, tabfm, exaone-tabular, limix-2"
Anomaly detectionanomaly_detection,In the SDK,"Rank rows by how unusual they are, with no labels: fraud, errors, rare cases. Says which columns make each row unusual.","tabpfn-v2, tabicl-v2, fomo-0d",—
Imputationimputation,In the SDK,"Fill in missing values from the rest of the row, after a championship on cells we hide.","tabpfn-v2, tabicl-v2",limix-2
Synthetic datasynthetic_data,In the SDK,"New rows with the same columns and relations as yours, scored by how hard they are to tell apart from the real ones.","tabpfn-v2, tabicl-v2",—
Clusteringclustering,In the SDK,"Group rows into segments with no labels, and describe what sets each segment apart.",zeus,—


## The data

Card transactions: the amount follows the merchant category and the hour, international ones are rarer, and 2% are fraud that breaks the
pattern (a big amount at a grocery store at 4 am).

In [3]:
rng = np.random.default_rng(0)
n = 1500
category = rng.choice(["grocery", "fuel", "travel", "electronics"], n, p=[0.45, 0.25, 0.15, 0.15])
base = pd.Series(category).map({"grocery": 40, "fuel": 55, "travel": 400, "electronics": 250}).to_numpy()
hour = np.clip(rng.normal(14, 4, n), 0, 23).round()
tx = pd.DataFrame({
    "category": category,
    "hour": hour,
    "amount": (base * rng.lognormal(0, 0.35, n)).round(2),
    "international": rng.random(n) < np.where(category == "travel", 0.6, 0.05),
    "fraud": 0,
})
bad = rng.choice(n, 30, replace=False)
tx.loc[bad, "amount"] = (tx.loc[bad, "amount"] * rng.uniform(8, 15, 30)).round(2)
tx.loc[bad, "hour"] = rng.integers(1, 5, 30)
tx.loc[bad, "fraud"] = 1
tx.head()

,category,hour,amount,international,fraud
0,fuel,15.0,58.08,False,0
1,grocery,15.0,29.53,False,0
2,grocery,13.0,32.62,False,0
3,grocery,12.0,42.01,False,0
4,travel,13.0,727.04,False,0


## 1. Anomaly detection

With known labels (`labels="fraud"`), every method is scored by how well it ranks the fraud first and the best one wins. Without labels, the
first foundation model scores the rows. The labels are never read as a feature.

In [4]:
found = zg.anomalies(tx, labels="fraud")
found.evaluation

✓ 1,500 rows · 4 columns used


… TabPFN v2 · scoring 1,500 rows


✓ TabPFN v2 scored 4m01s


… TabICLv2 · scoring 1,500 rows


✓ TabICLv2 scored 4m10s


… FoMo-0D · scoring 1,500 rows


... (13 outputs left out)

     without a model       0.500  (ROC AUC, random order)


◆ Champion: knn · ROC AUC 0.995 · Built with PriorLabs-TabPFN


◆ 75 unusual rows flagged by knn · top 5% · .top() shows them with why


,roc_auc,pr_auc,kind
model,,,
knn,0.994875,0.799111,classic
local-outlier-factor,0.992358,0.707399,classic
isolation-forest,0.981224,0.638657,classic
fomo-0d,0.697211,0.111971,foundation model
tabicl-v2,0.281973,0.014695,foundation model
tabpfn-v2,0.281610,0.014644,foundation model


Without labels, the first foundation model scores every row. TabPFN v2 and TabICLv2 also say which columns make each row unusual
(from the model's own per-column factors), in `why`:

In [5]:
unlabeled = zg.anomalies(tx.drop(columns="fraud"), models=["tabpfn-v2"])
unlabeled.top(10)

✓ 1,500 rows · 4 columns used


… TabPFN v2 · scoring 1,500 rows


✓ TabPFN v2 scored 3m53s


◆ 75 unusual rows flagged by tabpfn-v2 · top 5% · .top() shows them with why


,anomaly_score,why,category,hour,amount,international
934,15.177823,"international, category, amount",fuel,16.0,55.17,True
1050,15.060233,"international, category, amount",fuel,15.0,30.86,True
885,14.786484,"international, category, amount",fuel,12.0,53.88,True
304,14.762712,"international, category, amount",fuel,9.0,99.23,True
488,14.690974,"international, category, amount",fuel,17.0,30.36,True
1047,14.580599,"international, category, amount",fuel,19.0,43.68,True
162,14.576977,"international, category, amount",fuel,17.0,88.00,True
1117,14.532106,"international, category, amount",fuel,21.0,38.27,True
493,14.512976,"international, category, amount",fuel,11.0,43.03,True
191,14.505431,"international, category, amount",fuel,12.0,74.69,True


In [6]:
unlabeled.scores.loc[bad, "is_anomaly"].mean()    # share of the planted fraud it flagged, without ever seeing a label

np.float64(0.0)

Score new rows against a clean reference instead, for example today's transactions against last month's:

In [7]:
today = zg.anomalies(tx.drop(columns="fraud").iloc[:200], reference=tx[tx.fraud == 0].drop(columns="fraud").iloc[200:])
today.anomalies.head()

✓ 200 rows · 4 columns used


… TabPFN v2 · scoring 200 rows


✓ TabPFN v2 scored 1m16s


◆ 10 unusual rows flagged by tabpfn-v2 · top 5% · .top() shows them with why


,category,hour,amount,international
18,grocery,3.0,391.84,False
22,fuel,11.0,43.82,True
63,fuel,3.0,883.62,False
72,grocery,15.0,43.01,True
119,grocery,1.0,508.56,False


## 2. Imputation

Some cells go missing. `impute` first hides known cells and scores every method on them, then the champion fills the real gaps from the rest
of each row.

In [8]:
holes = tx.drop(columns="fraud").copy()
holes.loc[rng.choice(n, 150, replace=False), "amount"] = np.nan
holes.loc[rng.choice(n, 100, replace=False), "category"] = None
filled = zg.impute(holes)
filled.evaluation

✓ 1,500 rows · 250 missing cells in 2 columns · hiding 275 known cells to score the models


… TabPFN v2 · filling 275 hidden cells


✓ TabPFN v2 done 19.6s


… TabICLv2 · filling 275 hidden cells


✓ TabICLv2 done 18.3s


… mean-mode · filling 275 hidden cells


... (10 outputs left out)

◆ Champion: tabicl-v2 · imputation error 0.406 · Built with PriorLabs-TabPFN


… tabicl-v2 · filling 250 missing cells


✓ Gaps filled 17.3s


,error,error:category,error:amount,kind
model,,,,
tabicl-v2,0.405910,0.292857,0.518962,foundation model
tabpfn-v2,0.451549,0.300000,0.603098,foundation model
knn-imputer,0.614031,0.642857,0.585204,classic
mean-mode,0.778389,0.578571,0.978207,classic
iterative-imputer,0.804116,0.650000,0.958233,classic


In [9]:
filled.data[filled.filled.any(axis=1)].head()

,category,hour,amount,international
0,grocery,15.0,58.080000,False
4,travel,13.0,727.040000,False
9,electronics,14.0,271.587006,False
19,grocery,16.0,161.320465,False
27,grocery,8.0,94.638039,False


## 3. Synthetic data

New rows with the same columns and relations. A classifier then tries to tell them from the real ones: a detection AUC near 0.5 means it
can't. `exact_copies` is the share of synthetic rows that copy a real row.

In [10]:
fake = zg.synthesize(tx.drop(columns="fraud"), n=1000)
fake.evaluation

… TabPFN v2 · generating 1,000 rows


✓ TabPFN v2 done 49.6s


… TabICLv2 · generating 1,000 rows


✓ TabICLv2 done 48.1s


… independent-columns · generating 1,000 rows


✓ independent-columns done 0.0s


  🥇 tabicl-v2            ████████████████████████ 0.512


  🥈 tabpfn-v2            ███████████████████████ 0.521


  🥉 independent-columns  ████ 0.792


◆ Champion: tabicl-v2 · detection AUC (0.5 = indistinguishable) 0.512 · Built with PriorLabs-TabPFN


,detection_auc,exact_copies,kind
model,,,
tabicl-v2,0.512399,0.032,foundation model
tabpfn-v2,0.520516,0.033,foundation model
independent-columns,0.792168,0.022,classic


In [11]:
fake.data.groupby("category")["amount"].median().round(1), tx.groupby("category")["amount"].median().round(1)

(category
 electronics    237.4
 fuel            57.2
 grocery         41.8
 travel         418.0
 Name: amount, dtype: float64,
 category
 electronics    247.2
 fuel            56.5
 grocery         40.0
 travel         423.6
 Name: amount, dtype: float64)

## 4. Clustering

The wine dataset that ships with scikit-learn: 178 wines, 13 measurements, three cultivars. With `labels=` the known groups score each method
(adjusted Rand index); without, ZEUS picks the segments and `k` is chosen for you, or pass `k=3`.

In [12]:
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True).frame
groups = zg.cluster(wine, labels="target")
groups.evaluation

… ZEUS · grouping 178 rows


✓ ZEUS done 6.1s


… kmeans · grouping 178 rows


✓ kmeans done 0.1s


… gaussian-mixture · grouping 178 rows


✓ gaussian-mixture done 0.1s


... (2 outputs left out)

  🥉 zeus              ████ 0.866


◆ Champion: kmeans · adjusted Rand index 0.897


◆ 3 segments by kmeans · .profiles says what sets each apart


,k,kind,silhouette,adjusted_rand
model,,,,
kmeans,3,classic,0.284859,0.897495
gaussian-mixture,3,classic,0.284859,0.897495
zeus,5,foundation model,0.241091,0.865713


In [13]:
groups.profiles

,rows,share,what sets it apart
segment,,,
0,65,0.365,alcohol lower (-0.9 sd); color_intensity lower...
1,51,0.287,od280/od315_of_diluted_wines lower (-1.3 sd); ...
2,62,0.348,proline higher (+1.1 sd); flavanoids higher (+...


## Next

* The [evaluation notebook](https://github.com/zerogradlabs/zerograd-python/blob/main/examples/evaluation.ipynb): cross-validation by group and time, thresholds, importance and scikit-learn.
* Docs: [zerograd.dev/docs#no-target](https://zerograd.dev/docs#no-target).